# Projet IoT Anomaly Detection - Modèles Hybrides

## Mini-projet Master SIDI, Sécurité des systèmes informatiques
**Auteur** : RIZQY Mehdi et AIT MOHAMMED Mohamed
**Date** : Avril 2026  
**Dataset** : N-BaIoT (Danmini_Doorbell)

## 1. Installation et imports


In [13]:
# Cellule 1 : Téléchargement du dataset N-BaIoT
!wget -O N-BaIoT.zip "https://archive.ics.uci.edu/static/public/442/detection+of+iot+botnet+attacks+n+baiot.zip"

# Vérifier que le téléchargement est réussi
!ls -lh N-BaIoT.zip

--2026-04-18 20:53:49--  https://archive.ics.uci.edu/static/public/442/detection+of+iot+botnet+attacks+n+baiot.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘N-BaIoT.zip’

N-BaIoT.zip             [           <=>      ]   1.65G  50.7MB/s    in 89s     

2026-04-18 20:55:19 (18.9 MB/s) - ‘N-BaIoT.zip’ saved [1772922927]

-rw-r--r-- 1 root root 1.7G Apr 18 20:55 N-BaIoT.zip


In [14]:
# Cellule 2 : Extraction du dataset
import zipfile
import os

print("📦 Extraction en cours...")
with zipfile.ZipFile('N-BaIoT.zip', 'r') as zip_ref:
    zip_ref.extractall('N-BaIoT_dataset')
print("✅ Extraction terminée !")

# Vérifier le contenu extrait
print("\n📁 Structure du dossier :")
!ls -la N-BaIoT_dataset/

📦 Extraction en cours...
✅ Extraction terminée !

📁 Structure du dossier :
total 56
drwxr-xr-x 11 root root 4096 Apr 18 20:49 .
drwxr-xr-x  1 root root 4096 Apr 18 20:49 ..
drwxr-xr-x  2 root root 4096 Apr 18 20:50 Danmini_Doorbell
-rw-r--r--  1 root root 1776 Apr 18 20:55 demonstrate_structure.csv
drwxr-xr-x  2 root root 4096 Apr 18 20:49 Ecobee_Thermostat
drwxr-xr-x  2 root root 4096 Apr 18 20:49 Ennio_Doorbell
-rw-r--r--  1 root root 4626 Apr 18 20:55 N_BaIoT_dataset_description_v1.txt
drwxr-xr-x  2 root root 4096 Apr 18 20:49 Philips_B120N10_Baby_Monitor
drwxr-xr-x  2 root root 4096 Apr 18 20:49 Provision_PT_737E_Security_Camera
drwxr-xr-x  2 root root 4096 Apr 18 20:51 Provision_PT_838_Security_Camera
drwxr-xr-x  2 root root 4096 Apr 18 20:49 Samsung_SNH_1011_N_Webcam
drwxr-xr-x  2 root root 4096 Apr 18 20:49 SimpleHome_XCS7_1002_WHT_Security_Camera
drwxr-xr-x  2 root root 4096 Apr 18 20:50 SimpleHome_XCS7_1003_WHT_Security_Camera


In [15]:
# Cellule 3 : Explorer la structure des dossiers
import os

dataset_path = 'N-BaIoT_dataset'
print("📂 Arborescence :\n")
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # Limiter l'affichage des sous-dossiers profonds
        for file in files[:2]:
            print(f"{indent}  - {file}")

📂 Arborescence :

N-BaIoT_dataset/
  - N_BaIoT_dataset_description_v1.txt
  - demonstrate_structure.csv
  Danmini_Doorbell/
    - gafgyt_attacks.rar
    - junk.csv
  Provision_PT_838_Security_Camera/
    - gafgyt_attacks.rar
    - junk.csv
  Samsung_SNH_1011_N_Webcam/
    - gafgyt_attacks.rar
    - benign_traffic.csv
  Ennio_Doorbell/
    - gafgyt_attacks.rar
    - benign_traffic.csv
  Provision_PT_737E_Security_Camera/
    - gafgyt_attacks.rar
    - mirai_attacks.rar
  SimpleHome_XCS7_1002_WHT_Security_Camera/
    - gafgyt_attacks.rar
    - mirai_attacks.rar
  Ecobee_Thermostat/
    - gafgyt_attacks.rar
    - mirai_attacks.rar
  SimpleHome_XCS7_1003_WHT_Security_Camera/
    - gafgyt_attacks.rar
    - mirai_attacks.rar
  Philips_B120N10_Baby_Monitor/
    - gafgyt_attacks.rar
    - mirai_attacks.rar


## 2. Modifications globales pour tous les appareils

# 2.1. Lister tous les appareils

In [16]:
# Cellule 4 : Charger les données normales d'un appareil
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import subprocess
import glob

# Fonction pour extraire les fichiers .rar d’un appareil
def extract_rar_files(device_path):
    rar_files = glob.glob(os.path.join(device_path, "*.rar"))
    for rar in rar_files:
        subprocess.run(["unrar", "x", rar, device_path], capture_output=True)
    print(f"Extraction terminée pour {device_path}")

# Récupérer tous les dossiers d’appareils (sous-dossiers contenant benign_traffic.csv)
def get_device_folders(base_path):
    devices = []
    for item in os.listdir(base_path):
        item_path = os.path.join(base_path, item)
        if os.path.isdir(item_path):
            benign_file = os.path.join(item_path, "benign_traffic.csv")
            if os.path.exists(benign_file):
                devices.append(item_path)
    return devices

# Charger et préparer les données pour un appareil
def load_device_data(device_path):
    # Bénin
    benign_df = pd.read_csv(os.path.join(device_path, "benign_traffic.csv"))
    if 'label' in benign_df.columns:
        X_benign = benign_df.drop('label', axis=1)
        y_benign = benign_df['label']
    else:
        X_benign = benign_df
        y_benign = np.zeros(len(benign_df))

    # Attaques : tous les fichiers CSV sauf benign
    attack_files = glob.glob(os.path.join(device_path, "*.csv"))
    attack_files = [f for f in attack_files if "benign" not in os.path.basename(f).lower()]
    if not attack_files:
        # Si pas encore extrait, extraire les .rar
        extract_rar_files(device_path)
        attack_files = glob.glob(os.path.join(device_path, "*.csv"))
        attack_files = [f for f in attack_files if "benign" not in os.path.basename(f).lower()]

    attack_dfs = [pd.read_csv(f) for f in attack_files]
    attack_df = pd.concat(attack_dfs, ignore_index=True)
    if 'label' in attack_df.columns:
        X_attack_all = attack_df.drop('label', axis=1)
        y_attack_all = attack_df['label']
    else:
        X_attack_all = attack_df
        y_attack_all = np.ones(len(attack_df))

    # Alignement des colonnes
    common_cols = [col for col in X_benign.columns if col in X_attack_all.columns]
    X_attack_aligned = X_attack_all[common_cols]
    for col in X_benign.columns:
        if col not in X_attack_aligned.columns:
            X_attack_aligned[col] = 0
    X_benign = X_benign[common_cols]

    # Split normal
    X_train_norm, X_val_norm, y_train_norm, y_val_norm = train_test_split(
        X_benign, y_benign, test_size=0.33, random_state=42, stratify=y_benign
    )

    # Normalisation
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_norm)
    X_val = scaler.transform(X_val_norm)
    y_train = np.zeros(len(X_train))
    y_val = np.zeros(len(X_val))

    # Attaques
    X_attack_scaled = scaler.transform(X_attack_aligned)
    y_attack = y_attack_all[:len(X_attack_scaled)]

    # Split attaques
    X_attack_train, X_attack_test, y_attack_train, y_attack_test = train_test_split(
        X_attack_scaled, y_attack, test_size=0.8, random_state=42, stratify=y_attack
    )

    return (X_train, X_val, X_attack_train, X_attack_test,
            y_train, y_val, y_attack_train, y_attack_test, scaler)

## 2.2. Fonctions d’entraînement de Modèle H1 : Auto-Encoder + Random Forest et Modèle H2 : MLP + Random Forest

In [17]:
def train_H1(X_train, X_val, X_attack_train, X_attack_test, y_attack_train, y_attack_test):
    # Auto-encoder
    input_dim = X_train.shape[1]
    input_layer = layers.Input(shape=(input_dim,))
    encoded = layers.Dense(64, activation='relu')(input_layer)
    encoded = layers.Dense(32, activation='relu')(encoded)
    decoded = layers.Dense(64, activation='relu')(encoded)
    decoded = layers.Dense(input_dim, activation='linear')(decoded)
    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    autoencoder.fit(X_train, X_train, epochs=10, batch_size=256, validation_split=0.1, verbose=0)

    # Erreurs de reconstruction
    mse_train = np.mean(np.square(X_train - autoencoder.predict(X_train, verbose=0)), axis=1)
    mse_val = np.mean(np.square(X_val - autoencoder.predict(X_val, verbose=0)), axis=1)
    mse_attack_train = np.mean(np.square(X_attack_train - autoencoder.predict(X_attack_train, verbose=0)), axis=1)
    mse_attack_test = np.mean(np.square(X_attack_test - autoencoder.predict(X_attack_test, verbose=0)), axis=1)

    X_train_rf = np.concatenate([mse_train, mse_attack_train]).reshape(-1,1)
    y_train_rf = np.concatenate([np.zeros(len(mse_train)), np.ones(len(mse_attack_train))])
    X_test_rf = np.concatenate([mse_val, mse_attack_test]).reshape(-1,1)
    y_test_rf = np.concatenate([np.zeros(len(mse_val)), np.ones(len(mse_attack_test))])

    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train_rf, y_train_rf)
    y_pred = rf.predict(X_test_rf)
    return {
        'accuracy': accuracy_score(y_test_rf, y_pred),
        'precision': precision_score(y_test_rf, y_pred),
        'recall': recall_score(y_test_rf, y_pred),
        'f1': f1_score(y_test_rf, y_pred),
        'confusion': confusion_matrix(y_test_rf, y_pred)
    }

def train_H2(X_train, X_val, X_attack_train, X_attack_test, y_attack_train, y_attack_test):
    # Concaténation pour MLP
    X_train_mlp = np.vstack([X_train, X_attack_train])
    y_train_mlp = np.hstack([np.zeros(len(X_train)), np.ones(len(X_attack_train))])
    X_test_mlp = np.vstack([X_val, X_attack_test])
    y_test_mlp = np.hstack([np.zeros(len(X_val)), np.ones(len(X_attack_test))])

    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout
    mlp = Sequential([
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    mlp.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    mlp.fit(X_train_mlp, y_train_mlp, epochs=10, batch_size=256, validation_split=0.2, verbose=0)

    # Extraire les features avant dernière couche
    feature_extractor = Sequential()
    for layer in mlp.layers[:-1]:
        feature_extractor.add(layer)
    train_features = feature_extractor.predict(X_train_mlp, verbose=0)
    test_features = feature_extractor.predict(X_test_mlp, verbose=0)

    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(train_features, y_train_mlp)
    y_pred = rf.predict(test_features)
    return {
        'accuracy': accuracy_score(y_test_mlp, y_pred),
        'precision': precision_score(y_test_mlp, y_pred),
        'recall': recall_score(y_test_mlp, y_pred),
        'f1': f1_score(y_test_mlp, y_pred),
        'confusion': confusion_matrix(y_test_mlp, y_pred)
    }

# 2.3. Boucle sur tous les appareils et stockage des résultats

In [19]:
base_path = "N-BaIoT_dataset"
device_folders = get_device_folders(base_path)
results = []

for device_path in device_folders:
    device_name = os.path.basename(device_path)
    print(f"\n=== Traitement de {device_name} ===")
    (X_train, X_val, X_attack_train, X_attack_test,
     y_train, y_val, y_attack_train, y_attack_test, _) = load_device_data(device_path)

    print("Entraînement H1...")
    h1_metrics = train_H1(X_train, X_val, X_attack_train, X_attack_test, y_attack_train, y_attack_test)
    print("Entraînement H2...")
    h2_metrics = train_H2(X_train, X_val, X_attack_train, X_attack_test, y_attack_train, y_attack_test)

    results.append({
        'device': device_name,
        'H1_accuracy': h1_metrics['accuracy'],
        'H1_precision': h1_metrics['precision'],
        'H1_recall': h1_metrics['recall'],
        'H1_f1': h1_metrics['f1'],
        'H2_accuracy': h2_metrics['accuracy'],
        'H2_precision': h2_metrics['precision'],
        'H2_recall': h2_metrics['recall'],
        'H2_f1': h2_metrics['f1'],
        'H1_cm': h1_metrics['confusion'],
        'H2_cm': h2_metrics['confusion']
    })

# Affichage tableau récapitulatif
results_df = pd.DataFrame(results)[['device', 'H1_accuracy', 'H1_precision', 'H1_recall', 'H1_f1',
                                    'H2_accuracy', 'H2_precision', 'H2_recall', 'H2_f1']]
print("\n=== RÉSULTATS GLOBAUX ===")
print(results_df.to_string())


=== Traitement de Danmini_Doorbell ===
Entraînement H1...
Entraînement H2...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



=== Traitement de Provision_PT_838_Security_Camera ===
Entraînement H1...
Entraînement H2...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



=== Traitement de Samsung_SNH_1011_N_Webcam ===
Extraction terminée pour N-BaIoT_dataset/Samsung_SNH_1011_N_Webcam
Entraînement H1...
Entraînement H2...


...


=== Traitement de SimpleHome_XCS7_1003_WHT_Security_Camera ===
Extraction terminée pour N-BaIoT_dataset/SimpleHome_XCS7_1003_WHT_Security_Camera
Entraînement H1...
Entraînement H2...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



=== Traitement de Philips_B120N10_Baby_Monitor ===
Extraction terminée pour N-BaIoT_dataset/Philips_B120N10_Baby_Monitor
Entraînement H1...
Entraînement H2...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



=== RÉSULTATS GLOBAUX ===
                                     device  H1_accuracy  H1_precision  H1_recall     H1_f1  H2_accuracy  H2_precision  H2_recall     H2_f1
0                          Danmini_Doorbell     0.999795      0.999961   0.999824  0.999893     0.999926      0.999988   0.999934  0.999961
1          Provision_PT_838_Security_Camera     0.999482      0.999922   0.999503  0.999712     0.999837      1.000000   0.999820  0.999910
2                 Samsung_SNH_1011_N_Webcam     0.999307      0.999942   0.999319  0.999630     0.999775      0.999985   0.999776  0.999880
3                            Ennio_Doorbell     0.999692      0.999960   0.999716  0.999838     0.999868      1.000000   0.999862  0.999931
4         Provision_PT_737E_Security_Camera     0.999535      0.999965   0.999539  0.999752     0.999694      0.999997   0.999677  0.999837
5  SimpleHome_XCS7_1002_WHT_Security_Camera     0.999505      0.999952   0.999530  0.999741     0.999891      0.999997   0.999888  0.

# 3. Interface graphique utilisateur (GUI) avec ipywidgets


In [21]:
# Interface graphique attrayante avec métriques et matrice de confusion (version corrigée)

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Style CSS pour une présentation moderne
display(HTML("""
<style>
    .dashboard-container {
        background: linear-gradient(135deg, #f6f9fc 0%, #e9f2f9 100%);
        border-radius: 30px;
        padding: 25px;
        margin: 15px;
        box-shadow: 0 20px 35px rgba(0,0,0,0.1);
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    }
    .title {
        font-size: 2rem;
        font-weight: 800;
        text-align: center;
        background: linear-gradient(45deg, #0b3b5f, #1b6b8f);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        margin-bottom: 20px;
    }
    .card-container {
        display: flex;
        flex-wrap: wrap;
        justify-content: space-around;
        gap: 20px;
        margin: 30px 0;
    }
    .metric-card {
        background: white;
        border-radius: 20px;
        padding: 20px;
        width: 180px;
        text-align: center;
        box-shadow: 0 8px 20px rgba(0,0,0,0.08);
        transition: transform 0.2s, box-shadow 0.2s;
        cursor: default;
    }
    .metric-card:hover {
        transform: translateY(-5px);
        box-shadow: 0 15px 30px rgba(0,0,0,0.12);
    }
    .metric-value {
        font-size: 2.2rem;
        font-weight: bold;
        margin: 10px 0;
    }
    .metric-label {
        font-size: 0.9rem;
        color: #555;
        letter-spacing: 0.5px;
    }
    .device-model {
        text-align: center;
        font-size: 1.2rem;
        background: rgba(255,255,255,0.7);
        padding: 10px;
        border-radius: 50px;
        margin: 10px auto;
        width: fit-content;
    }
    .cm-container {
        margin-top: 20px;
        text-align: center;
    }
</style>
"""))

# Noms d'appareils plus lisibles avec icônes
device_display_names = {
    'Danmini_Doorbell': '🚪 Sonnette Danmini',
    'Ecobee_Thermostat': '🌡️ Thermostat Ecobee',
    'Ennio_Doorbell': '🔔 Sonnette Ennio',
    'Philips_B120N10_Baby_Monitor': '👶 Baby Monitor Philips',
    'Provision_PT_737E_Security_Camera': '📷 Caméra PT-737E',
    'Provision_PT_838_Security_Camera': '📷 Caméra PT-838',
    'Samsung_SNH_1011_N_Webcam': '📹 Webcam Samsung',
    'SimpleHome_XCS7_1002_WHT_Security_Camera': '🏠 Caméra SimpleHome XCS7-1002',
    'SimpleHome_XCS7_1003_WHT_Security_Camera': '🏠 Caméra SimpleHome XCS7-1003'
}

# Widgets stylisés
device_select = widgets.Dropdown(
    options=[(device_display_names.get(r['device'], r['device']), i) for i, r in enumerate(results)],
    description='📱 Appareil :',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='70%', margin='0 auto')
)

model_select = widgets.ToggleButtons(
    options=['🔧 Modèle H1 (AE + RF)', '🧠 Modèle H2 (MLP + RF)'],
    description='🤖 Modèle :',
    style={'description_width': 'initial'},
    button_style='primary',
    layout=widgets.Layout(width='70%', margin='0 auto')
)

output_area = widgets.Output(layout=widgets.Layout(padding='10px'))

def update_dashboard(change=None):
    with output_area:
        clear_output(wait=True)
        idx = device_select.value
        model_choice = model_select.value
        device_raw = results[idx]['device']
        device_name = device_display_names.get(device_raw, device_raw)

        # Sélection des métriques selon le modèle
        if 'H1' in model_choice:
            acc = results[idx]['H1_accuracy']
            prec = results[idx]['H1_precision']
            rec = results[idx]['H1_recall']
            f1 = results[idx]['H1_f1']
            cm = results[idx]['H1_cm']
            model_name = "Auto-Encoder + Random Forest"
            primary_color = "#1f77b4"
            cmap = 'Blues'
        else:
            acc = results[idx]['H2_accuracy']
            prec = results[idx]['H2_precision']
            rec = results[idx]['H2_recall']
            f1 = results[idx]['H2_f1']
            cm = results[idx]['H2_cm']
            model_name = "MLP + Random Forest"
            primary_color = "#2ca02c"
            cmap = 'Greens'

        # Construction des cartes de métriques
        cards_html = f"""
        <div class="card-container">
            <div class="metric-card">
                <div style="font-size:2rem;">✅</div>
                <div class="metric-value" style="color:{primary_color}">{acc:.4f}</div>
                <div class="metric-label">Accuracy</div>
            </div>
            <div class="metric-card">
                <div style="font-size:2rem;">🎯</div>
                <div class="metric-value" style="color:{primary_color}">{prec:.4f}</div>
                <div class="metric-label">Precision</div>
            </div>
            <div class="metric-card">
                <div style="font-size:2rem;">📞</div>
                <div class="metric-value" style="color:{primary_color}">{rec:.4f}</div>
                <div class="metric-label">Recall</div>
            </div>
            <div class="metric-card">
                <div style="font-size:2rem;">⭐</div>
                <div class="metric-value" style="color:{primary_color}">{f1:.4f}</div>
                <div class="metric-label">F1-Score</div>
            </div>
        </div>
        """

        # Affichage de la matrice de confusion avec matplotlib (plus fiable)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                    xticklabels=['Normal', 'Anomalie'],
                    yticklabels=['Normal', 'Anomalie'],
                    ax=ax, cbar=False)
        ax.set_title(f'Matrice de confusion', fontsize=14, fontweight='bold')
        ax.set_xlabel('Prédiction')
        ax.set_ylabel('Vérité terrain')
        plt.tight_layout()

        # Assemblage de l'affichage
        display(HTML(f"""
        <div class="dashboard-container">
            <div class="title">🔍 Détection d'anomalies IoT - Tableau de bord</div>
            <div class="device-model">
                📟 <b>{device_name}</b> &nbsp;|&nbsp; 🧪 <b>{model_name}</b>
            </div>
            {cards_html}
            <div class="cm-container">
                <h4>📊 Matrice de confusion</h4>
            </div>
        </div>
        """))
        plt.show()

# Observateurs
device_select.observe(update_dashboard, names='value')
model_select.observe(update_dashboard, names='value')

# Assemblage et affichage
dashboard = widgets.VBox([
    widgets.HBox([device_select], layout=widgets.Layout(justify_content='center')),
    widgets.HBox([model_select], layout=widgets.Layout(justify_content='center')),
    output_area
])

display(HTML("<h1 style='text-align:center; margin-bottom:0;'>📡 IoT Anomaly Detection Dashboard</h1>"))
display(dashboard)
update_dashboard()